In [1]:
import pandas as pd
import os
import csv
import sys

# Use this specific number instead of sys.maxsize
# 2147483647 is the maximum value for a 32-bit signed integer (safe for C long)
csv.field_size_limit(2147483647)

data_path = r"C:\Users\hello\OneDrive\Documents\Intern\Layer10_Project\data\email.csv"

if os.path.exists(data_path):
    # There are 517401 total rows so I have used all 
    df = pd.read_csv(data_path, nrows=517401, engine='python', on_bad_lines='skip')
    print(f"Successful {len(df)} rows.")
    display(df.head())
else:
    print(f"Error {data_path}")

Error C:\Users\hello\OneDrive\Documents\Intern\Layer10_Project\data\email.csv


In [5]:
# Change '0' to any number to see different emails
sample_email = df.iloc[0]['message']

print("--- FULL RAW EMAIL ---")
print(sample_email)

NameError: name 'df' is not defined

In [ ]:
def extract_actual_message(raw_email):
    parts = raw_email.split('\n\n', 1)
    if len(parts) > 1:
        return parts[1].strip() # This is your actual message
    return ""

df['body_only'] = df['message'].apply(extract_actual_message)
print(df['body_only'].iloc[0])

Here is our forecast


In [ ]:
long_emails = df[df['message'].str.len() > 1000]

if not long_emails.empty:
    # SAFE PICK: Pick the LAST long email found, no matter how many there are
    # index [-1] always grabs the last one safely
    target_row = long_emails.iloc[-1] 
    
    raw_content = target_row['message']
    
    # Split the body from headers
    actual_message = raw_content.split('\n\n', 1)[1]
    
    print(f" Success {len(long_emails)}")
    print("-" * 50)
    print(actual_message)
else:
    print("Still no emails longer than 1000 chars in this sample.")

✅ Success! Found a long email (363088 matches found).
--------------------------------------------------
i think the YMCA has a class that is for people recovering from heart-attacks
i remeber something about that

 -----Original Message-----
From: 	Livia_Zufferli@Monitor.com@ENRON  
Sent:	Monday, November 26, 2001 11:44 AM
To:	Zufferli, John
Subject:	RE: ali's essays


i don't know about the heart classes.  i'll look into it, but her dr (ravi)
isn't offering up any suggestions or anything.  she saw him before the
surgery in august, and he said things were okay.  i really don't think he's
too helpful.

she is lazy -- but it really frustrates me that she doesn't want to help
herself.  i told her that not walking is like not taking her heart
medication.  that didn't seem to resonate.  dad is going to go to the YMCA
tomorrow and maybe get a membership for both of them -- they have a walking
track there (at least it's something to do in the winter).  when she was
down this weekend, we walk

In [ ]:
# Expanded list of executive folders
key_execs = ['lay-k', 'skilling-j', 'allen-p', 'kaminski-v', 'dasovich-j', 'sanders-r']

# Created the subset
mask = df['file'].str.contains('|'.join(key_execs), case=False, na=False)
df_execs = df[mask]

final_subset = df_execs[df_execs['message'].str.len() > 500]

print(f"Success {len(final_subset)}")

if len(final_subset) > 0:
    print("\n--- Sample from the subset ---")
    print(final_subset.iloc[0]['message'].split('\n\n', 1)[1][:300] + "...")

✅ Success! Created a high-quality subset of 75201 emails.

--- Sample from the subset ---
Traveling to have a business meeting takes the fun out of the trip.  Especially if you have to prepare a presentation.  I would suggest holding the business plan meetings here then take a trip without any formal business meetings.  I would even try and get some honest opinions on whether a trip is e...


In [ ]:
# Save the high-quality subset to your project folder
final_subset.to_csv("enron_exec_subset.csv", index=False)
print("Saved")

✅ Saved! From now on, you can just load 'enron_exec_subset.csv'.


In [ ]:
import pandas as pd
import os

df = pd.read_csv("enron_exec_subset.csv")

# Ensure 'employee' column exists
if 'employee' not in df.columns:
    df['employee'] = df['file'].str.split('/').str[0]

# 1. THE EXECUTIVES (Top 500)
exec_folders = ['kaminski-v', 'dasovich-j', 'lay-k', 'skilling-j', 'sanders-r', 'jones-t']
df_execs = df[df['employee'].isin(exec_folders)]
# Sample 500 rows from these executives
sample_execs = df_execs.sample(n=min(500, len(df_execs)), random_state=42)

# 2. THE DIVERSE EMPLOYEES (The remaining 1500)
df_others = df[~df['employee'].isin(exec_folders)]

# We want "Informative" emails, so we filter for length > 800 characters
df_others_rich = df_others[df_others['message'].str.len() > 800]

# Sample 1500 from this diverse group
sample_others = df_others_rich.sample(n=min(1500, len(df_others_rich)), random_state=42)

final_balanced_set = pd.concat([sample_execs, sample_others])
final_balanced_set.to_csv("layer10_study_set.csv", index=False)

print(f"Balanced Dataset Created!")
print(f"Executive Emails: {len(sample_execs)}")
print(f"Other/Diverse Emails: {len(sample_others)}")
print(f"File saved as: layer10_study_set.csv")

✅ Balanced Dataset Created!
🔹 Executive Emails: 500
🔹 Other/Diverse Emails: 1500
📍 File saved as: layer10_study_set.csv
